In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [55]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

In [56]:
def get_exercise_type(row):
    tipos = [row.tipo_ejercicio_x, row.tipo_ejercicio_y]
    tipo  = [i for i in tipos if pd.notna(i)]
    if len(tipo) > 0:
        return tipo[0]
    else:
        return pd.NA

## Combinar enunciados con clasificaciones

Cargar enunciados

In [3]:
df_enun = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

Establecer `id` como columna índice, para poder hacer `update`

In [ ]:
df_enun = df_enun.set_index("id")

In [ ]:
df_enun.head(2)

In [ ]:
df_enun.columns

## Cargar ejercicios clasificados **a incorporar**

In [18]:
df_classified = pd.read_csv(root / "csv/ejercicios_clasificados_campo_electrico_magnetico_induccion.csv")

In [ ]:
df_classified.head()

Establecer `id` como columna índice, para poder hacer `update`

In [ ]:
df_classified = df_classified.set_index("id")

In [ ]:
df_classified.head(2)

In [ ]:
df_classified[df_classified.tipo_ejercicio.isna()]

### Inspeccionar clasificados

In [ ]:
df_classified.columns

In [ ]:
df_classified.sample()

Actualizar los ejercicios clasificados y resetear índice

In [ ]:
df_enun.update(df_classified.tipo_ejercicio)
df_enun = df_enun.reset_index(drop = False)

In [ ]:
rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','observaciones', 'enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
df_enun = df_enun[rearranged_cols]

Exportar enunciados clasificados actualizado

In [ ]:
df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
               index = False, encoding="utf-8")

In [ ]:
df_enun.query("asignatura == 'Física' and tema == 'campo eléctrico y magnético'").head(2)

## Split del tipo de ejercicio
Ejercicios con más de un tipo de ejercicio

In [23]:
df_enun.shape

(5235, 12)

In [20]:
df_enun.tipo_ejercicio = df_enun.tipo_ejercicio.str.split("|")

In [21]:
df_enun = df_enun.explode("tipo_ejercicio")

In [30]:
df_enun.reset_index(inplace=True)

In [22]:
df_enun.head(1)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,observaciones,enunciado,id,info_ejercicio,archivo,imagen
0,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,Verdadero o Falso,NaN,Discuta la veracidad de las siguientes afirmac...,fisica_campo_electrico_y_magnetico_2025_julio_...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...


In [31]:
df_enun[df_enun.asignatura == "Física"].tema.unique()

<ArrowStringArray>
['campo eléctrico y magnético',          'campo gravitatorio',
   'física cuántica y nuclear',                       'ondas',
           'óptica geométrica']
Length: 5, dtype: str

In [7]:
list(df_enun[(df_enun.asignatura == "Física") & (df_enun.tema == "campo eléctrico y magnético")].tipo_ejercicio.unique())

['Verdadero o Falso',
 'flujo magnético',
 'fuerza entre conductores',
 'campo magnético conductor rectilíneo',
 'distribución de cargas',
 'trayectoria campo magnético',
 'experimento de Henry',
 'acelerado diferencia de potencial',
 'campo magnético trayectoria',
 'fuerza campo eléctrico',
 'carga no se desvíe',
 'flujo magnético campo varía tiempo',
 'superficie equipotencial teoría',
 'flujo magnético ángulo varía tiempo',
 'campo eléctrico',
 'corriente inducida por imán',
 'potencial eléctrico',
 'inducción magnética',
 'campo eléctrico uniforme']

In [32]:
new_topic = {
"inducción magnética" : [
 'flujo magnético',
 'fuerza entre conductores',
 'experimento de Henry',
 'flujo magnético campo varía tiempo',
 'flujo magnético ángulo varía tiempo',
 'corriente inducida por imán',
 'inducción magnética',
 ],

"campo magnético" : [
 'campo magnético conductor rectilíneo',
 'acelerado diferencia de potencial',
 'campo magnético trayectoria',
 'carga no se desvíe',
 'trayectoria campo magnético',
 ],

 "campo eléctrico" : [
 'distribución de cargas',
 'fuerza campo eléctrico',
 'superficie equipotencial teoría',
 'potencial eléctrico',
 'campo eléctrico',
 'campo eléctrico uniforme'
 ]
}

In [33]:
df_enun.loc[df_enun.asignatura == "Física"].tema.unique()

<ArrowStringArray>
['campo eléctrico y magnético',          'campo gravitatorio',
   'física cuántica y nuclear',                       'ondas',
           'óptica geométrica']
Length: 5, dtype: str

In [38]:
elec_mag_ind_idx= df_enun[(df_enun.asignatura == "Física") & (df_enun.tema == "campo eléctrico y magnético")].index

In [39]:
elec_mag_ind_idx

RangeIndex(start=0, stop=216, step=1)

In [41]:
for idx in elec_mag_ind_idx:
    tipo_ejercicio = df_enun.loc[idx, "tipo_ejercicio"]
    found = False
    for tema, lista_tipos in new_topic.items():
        for tipo in lista_tipos:
            if tipo_ejercicio == tipo:
                df_enun.loc[idx, "tema"] = tema
                found = True
                break
        if found:
            break

In [51]:
df_enun[df_enun.asignatura == "Física"].tema.unique()

<ArrowStringArray>
[      'inducción magnética',           'campo magnético',
           'campo eléctrico',        'campo gravitatorio',
 'física cuántica y nuclear',                     'ondas',
         'óptica geométrica']
Length: 7, dtype: str

## Fix indexes after split

In [52]:
duplicated_indexes = df_enun[df_enun.duplicated(subset = ["id"])].index

In [53]:
duplicated_indexes

Index([  1,   3,   9,  11,  15,  18,  21,  24,  26,  28,  30,  32,  35,  37,
        39,  41,  45,  47,  49,  51,  53,  57,  58,  60,  63,  65,  67,  70,
        72,  74,  76,  78,  81,  83,  84,  86,  89,  90,  92,  94,  96,  98,
       100, 102, 103, 105, 107, 109, 111, 114, 116, 118, 120, 121, 124, 127,
       128, 130, 132, 134, 136, 138, 140, 142, 144, 147, 149, 150, 151, 153,
       155, 158, 160, 162, 166, 168, 171, 172, 174, 176, 180, 182, 184, 186,
       188, 190, 192, 194, 196, 199, 201, 203, 206, 209, 214],
      dtype='int64')

Añadir tipo ejercicio al id para ejercicios con varios tipos

In [57]:
for idx in duplicated_indexes:
    tipo_ejercicio = df_enun.loc[idx].tipo_ejercicio
    add_id = remove_accents(tipo_ejercicio).replace(" ", "_").lower()
    
    id = df_enun.loc[idx].id
    year_pos = id.find("2")
    new_id = id[:year_pos] + add_id + "_" + id[year_pos:]

    df_enun.loc[idx, "id"] = new_id  

In [65]:
df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
               index = False, encoding="utf-8")

Opcional: insertar enunciados limpios de matrices

In [ ]:
df_clean_st = pd.read_csv(root / "classifier/matrices_limpio.csv")

In [ ]:
df_clean_st.sample()

In [ ]:
df_enun = df_enun.set_index("id")
df_enun.update(df_clean_st.set_index("id").enunciado)

In [ ]:
df_enun[df_enun.tema == "matrices y determinantes"].sample(3)

Exportar enunciados clasificados actualizado

In [ ]:
df_enun.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv",
                                         index = False, encoding="utf-8")